# Atari REINFORCE with Nature CNN

This notebook shows the native REINFORCE workflow for Atari. It starts by checking that the ALE environments are registered, then wraps the base environment with `AtariPreprocessor`, and finally plugs `NatureCNNExtractor` into the policy.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[atari]"

In [ ]:
import gymnasium as gym

try:
    import ale_py
    print(f"ale_py successfully imported from: {ale_py.__file__}")
except ImportError:
    print("ale_py is not found or not importable.")
    
atari_env_ids = sorted(
    env_id for env_id in gym.registry.keys() if str(env_id).startswith('ALE/')
)
print(f'Registered ALE environments: {len(atari_env_ids)}')
print('Sample ids:', atari_env_ids[:12])

ENV_ID = 'ALE/Breakout-v5'
if ENV_ID not in atari_env_ids:
    raise RuntimeError(
        f'{ENV_ID} is not registered. Install the Atari extra before running this notebook.'
    )

## Why use `AtariPreprocessor`?

`AtariPreprocessor` handles the standard Atari preprocessing stack used by the examples in this package:

- grayscale conversion
- resize to `84 x 84`
- frame stacking as `(stack_size, height, width)`
- reward passthrough when `frame_skip=1`

Creating the base Gymnasium environment with `frameskip=1` avoids double frame skipping.
            

In [ ]:
import gymnasium as gym

from crosslearn import REINFORCE, make_vec_env
from crosslearn.envs import AtariPreprocessor
from crosslearn.extractors import NatureCNNExtractor

def make_env():
    env = gym.make(ENV_ID, render_mode='rgb_array', frameskip=1)
    return AtariPreprocessor(env, stack_size=4, frame_skip=1, screen_size=84)

sample_env = make_env()
print('Observation space:', sample_env.observation_space)
print('Action space:', sample_env.action_space)
sample_env.close()

In [ ]:
vec_env = make_vec_env(make_env, n_envs=1)

agent = REINFORCE(
    vec_env,
    n_steps=4096,
    features_extractor_class=NatureCNNExtractor,
    features_extractor_kwargs={'features_dim': 512},
    seed=42,
    verbose=2,
)
agent.learn(total_timesteps=100_000)      

## Evaluate a single episode

Atari usually needs much longer training than the short tutorial run above. This cell is mainly a smoke test that the preprocessing and policy wiring are correct.
            

In [ ]:
eval_env = make_env()
obs, info = eval_env.reset(seed=42)
terminated = False
truncated = False
episode_return = 0.0

while not (terminated or truncated):
    action = int(agent.predict(obs, deterministic=True))
    obs, reward, terminated, truncated, info = eval_env.step(action)
    episode_return += reward

print(f'Evaluation return on {ENV_ID}: {episode_return:.2f}')
eval_env.close()     